# 분류 모델 평가 실습

이 파일은 분류 모델의 평가 지표를 직접 확인하는 실습용 파일이다.

실습 구성
1. 이진 분류 평가 - 유방암 진단
2. 다중 분류 평가 - 아이리스

실습 목표
1. 분류 문제에서 train/test 분리 후 모델을 학습할 수 있다.
2. confusion matrix를 통해 어떤 예측이 맞고 틀렸는지 해석할 수 있다.
3. accuracy, precision, recall, f1-score의 의미를 결과와 함께 해석할 수 있다.
4. 이진 분류에서는 ROC Curve와 AUC를 확인할 수 있다.
5. 다중 분류에서는 classification report와 macro 평균의 의미를 확인할 수 있다.

진행 가이드
1. 각 문제의 요구사항을 먼저 읽는다.
2. 코드 셀은 직접 작성한다.
3. 출력 결과를 보고 해석 질문까지 답해본다.
4. 필요하면 셀을 추가해도 된다.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)


## 1. 이진 분류 평가 - 유방암 진단

실습 목적
- 악성(malignant) / 양성(benign) 두 클래스를 구분하는 이진 분류 문제를 다룬다.
- 정확도만 보는 것이 아니라 정밀도, 재현율, F1-score, ROC-AUC까지 함께 확인한다.
- 특히 의료 진단과 같이 놓치면 안 되는 문제에서 재현율이 왜 중요한지 생각해본다.

요구사항
1. `load_breast_cancer()`로 데이터를 불러온다.
2. 입력 데이터와 타깃 데이터의 shape를 확인한다.
3. DataFrame으로 변환하여 앞부분을 확인한다.
4. `train_test_split()`으로 학습/테스트 데이터를 분리한다.
   - `test_size=0.2`
   - `random_state=42`
   - `stratify=target`
5. `LogisticRegression` 모델을 학습한다.
6. 테스트 데이터 예측값과 양성 확률값을 구한다.
7. 아래 평가를 수행한다.
   - confusion matrix
   - accuracy
   - precision
   - recall
   - f1-score
   - classification report
8. ROC Curve를 그리고 AUC를 계산한다.

해석 질문
1. confusion matrix에서 FP와 FN은 각각 무엇을 의미하는가?
2. 의료 진단 문제에서 precision과 recall 중 어떤 값을 더 중요하게 볼 수 있는가?
3. AUC 값이 높다는 것은 무엇을 의미하는가?

In [1]:
# 1) 데이터 로드
from sklearn.datasets import load_breast_cancer

In [18]:
# 2) 데이터프레임으로 확인
bc = load_breast_cancer()
bc
bc_df = pd.DataFrame(bc.data, columns=bc.feature_names)
bc_df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [25]:
# 3) train / test 분리
X = bc_df
y = bc.target
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    random_state=42,
    test_size=0.2
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(455, 30) (455,)
(114, 30) (114,)


In [ ]:
# 4) 모델 학습
lr_clf = LogisticRegression(max_iter=10000)

lr_clf.fit(X_train, y_train)

print(lr_clf.score(X_train, y_train))
print(lr_clf.score(X_test, y_test))

0.9560439560439561
0.9649122807017544


In [29]:
# 5) 예측값 / 예측확률 구하기
print(lr_clf.predict(X_test))
print(lr_clf.predict_proba(X_test))

[0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1
 1 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0
 0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 1 0 1 0 1
 0 1 1]
[[1.00000000e+00 3.41501433e-11]
 [3.56449740e-05 9.99964355e-01]
 [9.49043246e-01 5.09567540e-02]
 [3.96461418e-01 6.03538582e-01]
 [9.99999998e-01 2.03502635e-09]
 [1.75004674e-02 9.82499533e-01]
 [3.06032155e-05 9.99969397e-01]
 [9.99990084e-01 9.91632037e-06]
 [9.99950675e-01 4.93247220e-05]
 [1.00000000e+00 1.46401719e-10]
 [1.37963473e-03 9.98620365e-01]
 [9.95063152e-01 4.93684814e-03]
 [9.04393720e-04 9.99095606e-01]
 [9.99988844e-01 1.11561165e-05]
 [9.99423465e-01 5.76534841e-04]
 [6.08973426e-02 9.39102657e-01]
 [7.78849226e-01 2.21150774e-01]
 [2.15233504e-02 9.78476650e-01]
 [1.44779389e-03 9.98552206e-01]
 [2.09401527e-02 9.79059847e-01]
 [9.96112983e-01 3.88701718e-03]
 [9.87583551e-01 1.24164486e-02]
 [3.47853541e-04 9.99652146e-01]
 [7.6466

In [30]:
# 6) confusion matrix 확인
print(confusion_matrix(y_test, lr_clf.predict(X_test)))

[[39  3]
 [ 1 71]]


In [ ]:
# 7) 평가지표 계산


In [ ]:
# 8) classification report 출력


In [ ]:
# 9) ROC Curve 시각화 및 AUC 계산


## 2. 다중 분류 평가 - 아이리스

실습 목적
- setosa, versicolor, virginica 세 품종을 구분하는 다중 분류 문제를 다룬다.
- 다중 분류에서는 클래스가 여러 개이므로 confusion matrix를 표 형태로 해석하는 연습이 중요하다.
- classification report에서 클래스별 precision, recall, f1-score를 비교해본다.

요구사항
1. `load_iris()`로 데이터를 불러온다.
2. 입력 데이터와 타깃 데이터의 shape를 확인한다.
3. `train_test_split()`으로 학습/테스트 데이터를 분리한다.
   - `test_size=0.2`
   - `random_state=42`
   - `stratify=y`
4. `LogisticRegression` 모델을 학습한다.
5. 테스트 데이터 예측값을 구한다.
6. confusion matrix를 출력하고 DataFrame 형태로 보기 좋게 정리한다.
7. classification report를 출력한다.
8. accuracy, precision, recall, f1-score를 `average='macro'`로 계산한다.

해석 질문
1. 어떤 클래스에서 오분류가 발생했는가?
2. macro average를 사용하는 이유는 무엇인가?
3. 이진 분류와 달리 다중 분류 평가에서 주의할 점은 무엇인가?


In [ ]:
# 1) 데이터 로드


In [ ]:
# 2) train / test 분리


In [ ]:
# 3) 모델 학습


In [ ]:
# 4) 예측값 구하기


In [ ]:
# 5) confusion matrix 출력


In [ ]:
# 6) confusion matrix를 DataFrame으로 정리


In [ ]:
# 7) classification report 출력


In [ ]:
# 8) macro 평균 기준 평가지표 계산


## 체크

1. 이진 분류에서 `predict()`와 `predict_proba()`를 구분해서 사용했는가?
2. ROC Curve의 x축이 FPR, y축이 TPR임을 확인했는가?
3. 다중 분류에서 precision, recall, f1-score 계산 시 `average` 옵션을 올바르게 사용했는가?
4. 결과 숫자만 출력하지 말고, 어떤 의미인지 해석까지 해보았는가?